# Generación de números pseudoaleatorios de 40 cifras

## 1. Objetivo

El objetivo de este trabajo es implementar tres técnicas para generar números pseudoaleatorios y analizar el comportamiento de los números obtenidos.

Las técnicas utilizadas son:

1. Generador Congruencial Lineal (LCG).
2. Generador basado en Gamma mediante SplitMix.
3. Generador basado en funciones Hash utilizando SHA-256.

Cada técnica recibe una semilla y sus respectivos parámetros. Por defecto, el programa genera 100.000 números de exactamente 40 cifras.

## 2. Generación de los números

Para garantizar que los números tengan 40 cifras, se utiliza el intervalo:

\[
10^{39} \leq N \leq 10^{40}-1
\]

De esta manera, todos los valores generados contienen exactamente 40 dígitos.

Los generadores producen bloques de bits que posteriormente se transforman al rango establecido. La semilla permite que las secuencias sean reproducibles: utilizando la misma semilla y los mismos parámetros se obtiene nuevamente la misma secuencia.

## 3. Técnicas utilizadas

### Generador Congruencial Lineal (LCG)

Utiliza la fórmula:

\[
X_{n+1}=(aX_n+c)\mod m
\]

Sus principales parámetros son la semilla, `a`, `c` y `m`.

### Generador Gamma / SplitMix

Utiliza una semilla y un parámetro denominado `gamma`. El estado se actualiza mediante operaciones sobre enteros de 64 bits y posteriormente se realizan operaciones de mezcla para obtener los valores de salida.

### Generador basado en Hash

Utiliza SHA-256 para producir los valores. Para cada número se combina la semilla con un contador y se aplica la función hash, obteniendo un bloque de datos que posteriormente se convierte en un número entero.

## 4. Comprobación de primalidad

Después de generar los números, se determina si cada uno es primo o no.

Debido a que los números tienen 40 cifras, no es eficiente intentar dividirlos entre todos los posibles divisores. Por esta razón, se utiliza la prueba de primalidad de **Miller-Rabin**, complementada con una comprobación inicial mediante números primos pequeños.

Miller-Rabin permite determinar rápidamente si un número es compuesto o si es probablemente primo.

## 5. Análisis de los resultados

Para cada ejecución se calculan:

- Cantidad de números generados.
- Cantidad de números distintos.
- Cantidad de números pares.
- Cantidad de números primos.
- Cantidad aproximada de primos esperados.
- Diferencia entre los primos encontrados y los esperados.
- Tiempo de generación.
- Tiempo utilizado para comprobar la primalidad.

También se guardan los números generados y su clasificación en un archivo de texto.

## 6. Aleatoriedad y reproducibilidad

Los algoritmos utilizados son **pseudoaleatorios**, lo que significa que utilizan procedimientos deterministas para generar una secuencia que presenta características similares a una secuencia aleatoria.

La semilla es importante porque determina la secuencia generada. Si se utiliza la misma semilla y los mismos parámetros, se puede reproducir la misma secuencia.

El hecho de utilizar números de 40 cifras permite trabajar con un espacio de valores muy grande y estudiar el comportamiento de los generadores con números enteros grandes. Sin embargo, tener más cifras no significa automáticamente que los números sean más aleatorios; la calidad depende principalmente del algoritmo y de sus parámetros.

## 7. Gobernanza de la información

Los resultados se almacenan en archivos para conservar un registro de los números generados y de su clasificación.

El uso de una semilla, los parámetros del algoritmo y el almacenamiento de los resultados permiten mantener la trazabilidad y reproducibilidad del experimento.

## 8. Conclusión

Se implementaron tres técnicas de generación pseudoaleatoria y se utilizaron para generar números de 40 cifras. Posteriormente, cada número fue analizado para determinar su primalidad mediante Miller-Rabin.

El experimento permite comprender la diferencia entre aleatoriedad verdadera y pseudoaleatoriedad, la importancia de las semillas y parámetros, y la necesidad de utilizar algoritmos eficientes cuando se trabaja con números enteros de gran tamaño.

In [ ]:
import hashlib
import math
import time


MINIMO = 10 ** 39
MAXIMO = 10 ** 40 - 1
RANGO = MAXIMO - MINIMO + 1

BITS_POR_NUMERO = 192

BASES_MILLER_RABIN = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37]


def bits_a_numero(bits):
    return MINIMO + (bits % RANGO)


def primos_hasta(limite):
    criba = [True] * (limite + 1)
    criba[0] = criba[1] = False

    for i in range(2, int(limite ** 0.5) + 1):
        if criba[i]:
            for j in range(i * i, limite + 1, i):
                criba[j] = False

    return [i for i, es in enumerate(criba) if es]


PRIMOS_PEQUENOS = primos_hasta(1000)


def es_primo(n):
    if n < 2:
        return False

    for p in PRIMOS_PEQUENOS:
        if n == p:
            return True
        if n % p == 0:
            return False

    d = n - 1
    r = 0

    while d % 2 == 0:
        d //= 2
        r += 1

    for a in BASES_MILLER_RABIN:
        x = pow(a, d, n)

        if x == 1 or x == n - 1:
            continue

        compuesto = True

        for _ in range(r - 1):
            x = (x * x) % n

            if x == n - 1:
                compuesto = False
                break

        if compuesto:
            return False

    return True


class Congruencial:

    nombre = "Congruencial lineal (LCG)"

    parametros_defecto = {
        "a": 6364136223846793005,
        "c": 1442695040888963407,
        "m": 2 ** 64
    }

    def __init__(self, semilla, a, c, m):
        self.x = semilla % m
        self.a = a
        self.c = c
        self.m = m

    def siguiente_bloque(self):
        self.x = (self.a * self.x + self.c) % self.m
        return self.x & 0xFFFFFFFFFFFFFFFF


class Gamma:

    nombre = "Weyl con gamma (SplitMix)"

    parametros_defecto = {
        "gamma": 0x9E3779B97F4A7C15
    }

    MASCARA = 0xFFFFFFFFFFFFFFFF

    def __init__(self, semilla, gamma):
        if gamma % 2 == 0:
            raise ValueError("gamma debe ser impar")

        self.estado = semilla & self.MASCARA
        self.gamma = gamma & self.MASCARA

    def siguiente_bloque(self):
        self.estado = (self.estado + self.gamma) & self.MASCARA

        z = self.estado
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & self.MASCARA
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & self.MASCARA
        z = z ^ (z >> 31)

        return z


class Hash:

    nombre = "Basado en hash"

    parametros_defecto = {
        "algoritmo": "sha256"
    }

    def __init__(self, semilla, algoritmo):
        self.semilla = semilla
        self.algoritmo = algoritmo
        self.contador = 0

    def siguiente_bloque(self):
        self.contador += 1

        mensaje = f"{self.semilla}:{self.contador}".encode()
        resumen = hashlib.new(self.algoritmo, mensaje).digest()

        return int.from_bytes(resumen[:8], "big")


TECNICAS = {
    "1": Congruencial,
    "2": Gamma,
    "3": Hash,
}


def generar(tecnica, semilla, cantidad, parametros):
    motor = tecnica(semilla, **parametros)
    bloques = BITS_POR_NUMERO // 64

    numeros = []

    for _ in range(cantidad):
        bits = 0

        for _ in range(bloques):
            bits = (bits << 64) | motor.siguiente_bloque()

        numeros.append(bits_a_numero(bits))

    return numeros


def analizar_primalidad(numeros):
    return [es_primo(n) for n in numeros]


def guardar(numeros, primalidad, ruta):
    with open(ruta, "w") as archivo:
        archivo.write("numero,es_primo\n")

        for n, p in zip(numeros, primalidad):
            archivo.write(f"{n},{'PRIMO' if p else 'NO PRIMO'}\n")


def pedir_entero(mensaje, valor_defecto):
    texto = input(f"{mensaje} [{valor_defecto}]: ").strip()
    return int(texto) if texto else valor_defecto


def pedir_parametros(tecnica):
    print(f"\nParametros de {tecnica.nombre}:")
    print("(deja vacio para usar el valor por defecto)")

    parametros = {}

    for nombre, defecto in tecnica.parametros_defecto.items():
        texto = input(f"   {nombre} [{defecto}]: ").strip()

        if not texto:
            parametros[nombre] = defecto
        elif isinstance(defecto, str):
            parametros[nombre] = texto
        else:
            parametros[nombre] = int(texto, 0)

    return parametros


def menu():
    print("=" * 60)
    print("GENERADOR DE NUMEROS PSEUDOALEATORIOS DE 40 CIFRAS")
    print("=" * 60)

    for clave, tecnica in TECNICAS.items():
        print(f"   {clave}. {tecnica.nombre}")

    opcion = input("\nEscoge una tecnica: ").strip()

    if opcion not in TECNICAS:
        print("Opcion no valida.")
        return

    tecnica = TECNICAS[opcion]

    semilla = pedir_entero("\nSemilla", 12345)
    cantidad = pedir_entero("Cuantos numeros", 100000)

    parametros = pedir_parametros(tecnica)

    print(f"\nGenerando {cantidad} numeros con {tecnica.nombre} ...")
    inicio = time.time()

    numeros = generar(tecnica, semilla, cantidad, parametros)

    tiempo_generar = time.time() - inicio
    print(f"Generados en {tiempo_generar:.2f} s")

    print("Probando primalidad ...")
    inicio = time.time()

    primalidad = analizar_primalidad(numeros)

    tiempo_primos = time.time() - inicio
    print(f"Probados en {tiempo_primos:.2f} s")

    encontrados = sum(primalidad)
    esperados = cantidad / math.log(10 ** 40)

    print("\nPrimeros 5 numeros:")
    for n, p in list(zip(numeros, primalidad))[:5]:
        print(f"   {n}   {'PRIMO' if p else 'NO PRIMO'}")

    cifras = {len(str(n)) for n in numeros}

    print(f"\nCifras por numero  : {sorted(cifras)}")
    print(f"Numeros distintos  : {len(set(numeros))} de {len(numeros)}")
    print(f"Numeros pares      : {sum(1 for n in numeros if n % 2 == 0)}")
    print(f"Primos encontrados : {encontrados}")
    print(f"Primos esperados   : {esperados:.0f}")
    print(f"Diferencia         : {(encontrados - esperados) / esperados * 100:+.2f} %")

    ruta = f"numeros_{tecnica.__name__.lower()}_{semilla}.txt"
    guardar(numeros, primalidad, ruta)
    print(f"\nGuardados en: {ruta}")


if __name__ == "__main__":
    menu()

GENERADOR DE NUMEROS PSEUDOALEATORIOS DE 40 CIFRAS
   1. Congruencial lineal (LCG)
   2. Weyl con gamma (SplitMix)
   3. Basado en hash

Escoge una tecnica: LCG
Opcion no valida.
